# Task4 — EfficientNet seed diversity

ต่อยอดสูตร Public 0.81047 ด้วย EfficientNet-B0 อีก seed (**2026**) คง augmentation, epochs, LR และ balancing เดิม เปลี่ยนการสุ่ม head initialization, ลำดับ batch และ augmentation. Pretrained backbone เริ่มจาก ImageNet weights ชุดเดิม ไม่เริ่มจาก full/fold checkpoint ที่ฝึกเสร็จแล้ว

**หนึ่งโมเดลใหม่ × 5 folds × 18 epochs** และ full refit อีก 18 epochs เฉพาะเมื่อ development/audit ผ่าน. Run All จะไม่เปิด Test หรือสร้าง submission

ทดสอบ baseline เดิม, seed2026 เดี่ยว และ **seed_bag50**: ResNet สองตัวตัวละ 25%, EfficientNet seed42/2026 ตัวละ 25%. รักษาน้ำหนักรวมตระกูล EfficientNet ไว้ที่ 50% ตาม weight study เดิม ไม่ค้นน้ำหนักเพิ่มเติม

เหตุผล: Fourier เพิ่มไม่สม่ำเสมอ แต่ EfficientNet เสริม ResNet ได้จริงรอบก่อน นี่เป็นการทดสอบว่าความต่างจาก seed ช่วยลด shared errors ได้หรือไม่ ไม่รับรองว่าจะเพิ่ม Public score. ชุด OOF/audit ถูกใช้หลายครั้งแล้ว ผลเป็น exploratory.

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_seed_training as training
from Task4.models import image_seed_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/seed_diversity_v1'
BATCH_SIZE=32
NUM_WORKERS=0
RUN_FULL_REFIT=True
NAME='efficientnet_seed2026_18'
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')
display(asdict(training.RECIPES[NAME]))

Device: mps


{'architecture': 'efficientnet_b0',
 'epochs': 18,
 'backbone_lr': 0.0001,
 'head_lr': 0.0005,
 'weight_decay': 0.0001,
 'schedule': 'cosine',
 'warmup_epochs': 1,
 'augmentation': 'compound_corruption',
 'label_smoothing': 0.0,
 'seed': 2026,
 'balance': 'weighted_loss',
 'jsd_weight': 0.0,
 'compound_probability': 0.6}

## 1. ตรวจ Train, folds และ reference checkpoints
ใช้ split เดิม seed42; เปลี่ยนเฉพาะ training seed เป็น 2026. เก็บผลใน run ใหม่ ไม่ทับ baseline.

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':asdict(training.RECIPES[NAME]),'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. ฝึกและประเมิน 5 folds

เซลล์ฝึกมีเพียง **หนึ่งเซลล์** วนครบทั้งห้า folds. checkpoint ทุก epoch; interruption กลาง epoch จะเริ่ม epoch นั้นใหม่. Fold ที่เสร็จแล้วโหลด cache

จากการรัน EfficientNet รอบก่อน ~205–210 วินาที/epoch: CV train อย่างเดียวประมาณ 5.1–5.3 ชั่วโมง; full Train เพิ่มประมาณ 78 นาทีหากผ่าน ยังไม่รวม original/stress/audit inference. เป็นประมาณการจากเครื่องเดิม ไม่ใช่เวลารับรอง

รายงาน baseline/seed2026/seed_bag50 โดยใช้ภาพ validation และ corrupted pixels เดิมกับ reference ทุกตัว ไม่มี Fourier augmentation ในรอบนี้

In [3]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet seed2026 fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet seed2026 fold 1/5
fold1 epoch 1 / 18 loss 0.74474 seconds 246.6
fold1 epoch 2 / 18 loss 0.38386 seconds 241.6
fold1 epoch 3 / 18 loss 0.28865 seconds 236.3
fold1 epoch 4 / 18 loss 0.24285 seconds 234.9
fold1 epoch 5 / 18 loss 0.20463 seconds 238.5
fold1 epoch 6 / 18 loss 0.18032 seconds 245.3
fold1 epoch 7 / 18 loss 0.15584 seconds 237.1
fold1 epoch 8 / 18 loss 0.1417 seconds 234.3
fold1 epoch 9 / 18 loss 0.12523 seconds 212.8
fold1 epoch 10 / 18 loss 0.11204 seconds 208.2
fold1 epoch 11 / 18 loss 0.10151 seconds 208.6
fold1 epoch 12 / 18 loss 0.09396 seconds 208.6
fold1 epoch 13 / 18 loss 0.08409 seconds 209.0
fold1 epoch 14 / 18 loss 0.08478 seconds 208.6
fold1 epoch 15 / 18 loss 0.0816 seconds 208.7
fold1 epoch 16 / 18 loss 0.07294 seconds 208.7
fold1 epoch 17 / 18 loss 0.07397 seconds 208.6
fold1 epoch 18 / 18 loss 0.07271 seconds 208.4
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2

suite,compound,original,single
procedure,,,
baseline,0.944725,0.986331,0.977201
seed2026,0.940351,0.979402,0.975016
seed_bag50,0.944788,0.985897,0.977794


Approx remaining CV TRAIN minutes (excludes validation): 250.3

EfficientNet seed2026 fold 2/5
fold2 epoch 1 / 18 loss 0.74617 seconds 208.8
fold2 epoch 2 / 18 loss 0.38381 seconds 207.9
fold2 epoch 3 / 18 loss 0.29402 seconds 208.5
fold2 epoch 4 / 18 loss 0.24756 seconds 208.4
fold2 epoch 5 / 18 loss 0.20588 seconds 208.4
fold2 epoch 6 / 18 loss 0.17734 seconds 208.6
fold2 epoch 7 / 18 loss 0.16227 seconds 208.4
fold2 epoch 8 / 18 loss 0.13572 seconds 208.4
fold2 epoch 9 / 18 loss 0.12261 seconds 208.1
fold2 epoch 10 / 18 loss 0.11471 seconds 208.7
fold2 epoch 11 / 18 loss 0.10677 seconds 208.5
fold2 epoch 12 / 18 loss 0.09448 seconds 208.5
fold2 epoch 13 / 18 loss 0.08236 seconds 208.6
fold2 epoch 14 / 18 loss 0.08785 seconds 208.7
fold2 epoch 15 / 18 loss 0.08032 seconds 208.3
fold2 epoch 16 / 18 loss 0.07487 seconds 208.3
fold2 epoch 17 / 18 loss 0.06773 seconds 208.4
fold2 epoch 18 / 18 loss 0.07146 seconds 208.0
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle

suite,compound,original,single
procedure,,,
baseline,0.955820,0.983375,0.985600
seed2026,0.953747,0.980941,0.984567
seed_bag50,0.954830,0.984802,0.985605


Approx remaining CV TRAIN minutes (excludes validation): 187.4

EfficientNet seed2026 fold 3/5
fold3 epoch 1 / 18 loss 0.74644 seconds 208.7
fold3 epoch 2 / 18 loss 0.38493 seconds 208.9
fold3 epoch 3 / 18 loss 0.29158 seconds 208.6
fold3 epoch 4 / 18 loss 0.24664 seconds 208.5
fold3 epoch 5 / 18 loss 0.20302 seconds 208.6
fold3 epoch 6 / 18 loss 0.17261 seconds 208.9
fold3 epoch 7 / 18 loss 0.15241 seconds 208.3
fold3 epoch 8 / 18 loss 0.13847 seconds 208.9
fold3 epoch 9 / 18 loss 0.12333 seconds 208.6
fold3 epoch 10 / 18 loss 0.10933 seconds 208.9
fold3 epoch 11 / 18 loss 0.09949 seconds 208.8
fold3 epoch 12 / 18 loss 0.09368 seconds 208.7
fold3 epoch 13 / 18 loss 0.08565 seconds 209.5
fold3 epoch 14 / 18 loss 0.0786 seconds 208.1
fold3 epoch 15 / 18 loss 0.07267 seconds 207.8
fold3 epoch 16 / 18 loss 0.06946 seconds 208.6
fold3 epoch 17 / 18 loss 0.07025 seconds 208.4
fold3 epoch 18 / 18 loss 0.06855 seconds 208.7
stress fold3 gaussian_1
stress fold3 gaussian_2
stress fold3 speckle_

suite,compound,original,single
procedure,,,
baseline,0.948277,0.984449,0.981614
seed2026,0.939226,0.979581,0.972767
seed_bag50,0.951450,0.984897,0.980606


Approx remaining CV TRAIN minutes (excludes validation): 125.1

EfficientNet seed2026 fold 4/5
fold4 epoch 1 / 18 loss 0.74315 seconds 208.3
fold4 epoch 2 / 18 loss 0.38726 seconds 208.2
fold4 epoch 3 / 18 loss 0.29375 seconds 208.2
fold4 epoch 4 / 18 loss 0.2356 seconds 208.8
fold4 epoch 5 / 18 loss 0.20978 seconds 208.3
fold4 epoch 6 / 18 loss 0.17412 seconds 208.5
fold4 epoch 7 / 18 loss 0.15844 seconds 208.7
fold4 epoch 8 / 18 loss 0.13468 seconds 209.0
fold4 epoch 9 / 18 loss 0.1204 seconds 208.8
fold4 epoch 10 / 18 loss 0.11283 seconds 208.9
fold4 epoch 11 / 18 loss 0.10424 seconds 208.9
fold4 epoch 12 / 18 loss 0.09662 seconds 208.8
fold4 epoch 13 / 18 loss 0.0875 seconds 209.0
fold4 epoch 14 / 18 loss 0.08164 seconds 208.4
fold4 epoch 15 / 18 loss 0.073 seconds 208.8
fold4 epoch 16 / 18 loss 0.07676 seconds 208.9
fold4 epoch 17 / 18 loss 0.07195 seconds 208.9
fold4 epoch 18 / 18 loss 0.07011 seconds 208.9
stress fold4 gaussian_1
stress fold4 gaussian_2
stress fold4 speckle_1
st

suite,compound,original,single
procedure,,,
baseline,0.942175,0.980890,0.981004
seed2026,0.939843,0.979322,0.970376
seed_bag50,0.943163,0.982340,0.980791


Approx remaining CV TRAIN minutes (excludes validation): 62.7

EfficientNet seed2026 fold 5/5
fold5 epoch 1 / 18 loss 0.75262 seconds 208.9
fold5 epoch 2 / 18 loss 0.38317 seconds 208.4
fold5 epoch 3 / 18 loss 0.29063 seconds 208.3
fold5 epoch 4 / 18 loss 0.22812 seconds 208.9
fold5 epoch 5 / 18 loss 0.20382 seconds 208.5
fold5 epoch 6 / 18 loss 0.1796 seconds 208.7
fold5 epoch 7 / 18 loss 0.15658 seconds 208.5
fold5 epoch 8 / 18 loss 0.13791 seconds 208.7
fold5 epoch 9 / 18 loss 0.12437 seconds 208.9
fold5 epoch 10 / 18 loss 0.10694 seconds 209.2
fold5 epoch 11 / 18 loss 0.10321 seconds 209.2
fold5 epoch 12 / 18 loss 0.09569 seconds 208.9
fold5 epoch 13 / 18 loss 0.08637 seconds 209.1
fold5 epoch 14 / 18 loss 0.07996 seconds 209.0
fold5 epoch 15 / 18 loss 0.07708 seconds 209.0
fold5 epoch 16 / 18 loss 0.07134 seconds 209.1
fold5 epoch 17 / 18 loss 0.07337 seconds 209.2
fold5 epoch 18 / 18 loss 0.07018 seconds 209.0
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1

suite,compound,original,single
procedure,,,
baseline,0.940842,0.984526,0.981418
seed2026,0.939330,0.979968,0.979588
seed_bag50,0.941802,0.984383,0.982417


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.55


## 3. เปรียบเทียบและเลือกสูตร

คง screening rules เดิม: original mean ลดไม่เกิน .001; family-balanced stress เพิ่มอย่างน้อย .002 และชนะ ≥4/5 folds; per-class ลดไม่เกิน .01; condition ลดไม่เกิน .02. หากกำไรน้อย/ไม่สม่ำเสมอ เก็บ baseline และไม่สร้าง submission ใหม่

ค่าเหล่านี้เป็น project heuristics ไม่ใช่ statistical significance. ไม่ปรับเกณฑ์ตามผลหลังรัน

In [4]:
display(evaluation.summarize(all_scores))
display(all_overlap[all_overlap.condition.eq('original')])
selection=evaluation.select(all_scores)
validation.write_json(OUT/'development_selection.json',selection)
display(selection)
# Check seed diversity against the original EfficientNet on identical OOF rows.
seed_rows=[]
for fold,frame in oof.items():
    original=pd.read_parquet(ROOT/'Task4/models/results/backbone_diversity_v1/efficientnet_compound18'/f'fold{fold}'/'oof.parquet')
    assert original.id.tolist()==frame.id.tolist() and np.array_equal(original.true,frame.true)
    a=training.probabilities(original,'tta').argmax(1)
    b=training.probabilities(frame,'tta').argmax(1);y=frame.true.to_numpy()
    seed_rows.append({'fold':fold,'rows':len(y),'old_seed_errors':int((a!=y).sum()),'new_seed_errors':int((b!=y).sum()),
                     'both_wrong':int(((a!=y)&(b!=y)).sum()),'new_seed_fixes':int(((a!=y)&(b==y)).sum()),
                     'new_seed_breaks':int(((a==y)&(b!=y)).sum()),'disagreements':int((a!=b).sum())})
seed_overlap=pd.DataFrame(seed_rows)
seed_overlap.to_parquet(OUT/'seed_error_overlap.parquet',index=False)
display(seed_overlap)

suite,compound,original,single
procedure,,,
baseline,0.946368,0.983914,0.981367
seed2026,0.942499,0.979843,0.976463
seed_bag50,0.947207,0.984464,0.981443


,fold,phase,condition,rows,baseline_errors,candidate_errors,both_wrong,candidate_fixes,candidate_breaks
0,1,development,original,6310,70,109,51,19,58
17,2,development,original,6309,88,106,68,20,38
34,3,development,original,6309,86,109,67,19,42
51,4,development,original,6309,96,109,71,25,38
68,5,development,original,6309,81,105,62,19,43


{'chosen': 'baseline',
 'reports': {'seed2026': {'original_gain': -0.004071343475574385,
   'stress_family_gain': -0.0047320055341944535,
   'stress_fold_wins': 0,
   'worst_class_gain': -0.0049831718771004676,
   'worst_condition_gain': -0.014214019562044888,
   'passed': False},
  'seed_bag50': {'original_gain': 0.0005498179173694995,
   'stress_family_gain': 0.0002023709986608354,
   'stress_fold_wins': 2,
   'worst_class_gain': -4.0550414878604e-06,
   'worst_condition_gain': -0.002797532597462138,
   'passed': False}},
 'rule': 'original >= -0.001; family-balanced stress >= +0.002; wins >=4/5; per-class >= -0.01; condition >= -0.02',
 'exploratory_reused_oof': True}

,fold,rows,old_seed_errors,new_seed_errors,both_wrong,new_seed_fixes,new_seed_breaks,disagreements
0,1,6310,95,109,61,34,48,89
1,2,6309,98,106,69,29,37,77
2,3,6309,120,109,79,41,30,84
3,4,6309,105,109,71,34,38,81
4,5,6309,97,105,73,24,32,65


## 4. Audit ของสูตรที่เลือก
Audit ใช้ veto เท่านั้น ไม่เปลี่ยนสูตรตาม audit. ไม่ใช่ untouched holdout เพราะเคยใช้ในงานก่อนหน้า

In [5]:
audit_result={'passed':False,'reason':'Development did not select a candidate'}
if selection['chosen']!='baseline':
    audit_parts=[]
    for fold in range(1,6):
        score,_=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold],phase='audit')
        audit_parts.append(score)
    audit_scores=pd.concat(audit_parts,ignore_index=True)
    audit_scores.to_parquet(OUT/'audit_scores.parquet',index=False)
    audit_result=evaluation.audit(audit_scores,selection['chosen'])
    display(evaluation.summarize(audit_scores))
validation.write_json(OUT/'audit_decision.json',audit_result)
display(audit_result)

{'passed': False, 'reason': 'Development did not select a candidate'}

## 5. Full refit และบันทึก readiness

ฝึก seed2026 บน Train ทั้งหมด 18 epochs เมื่อผ่าน development และ audit. ถ้าเลือก seed_bag50 ต้องใช้ full models เดิมอีกสามตัวร่วมด้วย โดยให้น้ำหนักเท่ากันตัวละ .25

ยังไม่เปิด Test/สร้าง submission ให้ตรวจผลก่อน. ไม่ต้องกด save weights เอง: fit_fixed บันทึก model.pt หลังจบ และ resume.pt ทุก epoch

In [6]:
full_path=None
if RUN_FULL_REFIT and selection['chosen']!='baseline' and audit_result['passed']:
    full_path=training.fit_fixed(RAW,groups,training.RECIPES[NAME],OUT/NAME/'full',BATCH_SIZE,NUM_WORKERS)
    validation.write_json(OUT/'ready_for_review.json',{
        'procedure':selection['chosen'],'seed_model':str(full_path),'sha256':validation.sha(full_path),
        'test_used':False,'submission_created':False,
        'selection':selection,'audit':audit_result})
    print('Saved full model:',full_path)
else:
    print('No full refit: inspect development/audit results above. Existing baseline preserved.')

No full refit: inspect development/audit results above. Existing baseline preserved.
